In [ ]:
import openai
from typing import Literal
from pydantic import BaseModel, create_model
from tqdm import tqdm

In [ ]:
x = ['f', 'l', 'p']

class Response(BaseModel):
    response:  Literal[tuple(x)]

In [ ]:
# Use this dataset instead though please (just a sampling)

In [ ]:
curr_path = "/workspace/psychometrics_for_LLMs/llm_psychometrics/data"
filename = "human_annotations_uuid.json"

In [ ]:
import json
with open(f"{curr_path}/human_annotations_uuid.json", "r") as f_in:
    targets = json.load(f_in)

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'YOUR_OPEN_AI_KEY_HERE'

In [ ]:
client = openai.Client()
response = client.responses.parse(
    model="gpt-4o-2024-08-06",
    input=[
        {"role": "system", "content": "Help users"},
        {
            "role": "user",
            "content": "Letter after k",
        },
    ],
    text_format=Response,
)

In [ ]:
response.output_parsed

In [ ]:
import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

In [ ]:
from datasets import load_dataset

# Load the dataset from HuggingFace
dataset = load_dataset(
    "thoughtworks/psychometric_personas",
    data_files="data/train-00000-of-00001.parquet",
    split="train"
)

# Optionally, display the first few rows to verify loading
#print(dataset[17])
dataset = dataset.filter(lambda x: x['uuid'] in targets)

In [ ]:
len(dataset)

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, conint, create_model

# Move all Pydantic model definitions to the top-level (not inside the loop)
class HonestyHumility(BaseModel):
    sincerity: conint(ge=0, le=5)
    fairness: conint(ge=0, le=5)
    greed_avoidance: conint(ge=0, le=5)
    modesty: conint(ge=0, le=5)

class Emotionality(BaseModel):
    fearfulness: conint(ge=0, le=5)
    anxiety: conint(ge=0, le=5)
    dependence: conint(ge=0, le=5)
    sentimentality: conint(ge=0, le=5)

class Extraversion(BaseModel):
    social_self_esteem: conint(ge=0, le=5)
    social_boldness: conint(ge=0, le=5)
    sociability: conint(ge=0, le=5)
    liveliness: conint(ge=0, le=5)

class Agreeableness(BaseModel):
    forgiveness: conint(ge=0, le=5)
    gentleness: conint(ge=0, le=5)
    flexibility: conint(ge=0, le=5)
    patience: conint(ge=0, le=5)

class Conscientiousness(BaseModel):
    organization: conint(ge=0, le=5)
    diligence: conint(ge=0, le=5)
    perfectionism: conint(ge=0, le=5)
    prudence: conint(ge=0, le=5)

class OpennessToExperience(BaseModel):
    aesthetic_appreciation: conint(ge=0, le=5)
    inquisitiveness: conint(ge=0, le=5)
    creativity: conint(ge=0, le=5)
    unconventionality: conint(ge=0, le=5)

class HexacoProfile(BaseModel):
    honesty_humility_score: conint(ge=0, le=5)
    honesty_humility: HonestyHumility
    emotionality_score: conint(ge=0, le=5)
    emotionality: Emotionality
    extraversion_score: conint(ge=0, le=5)
    extraversion: Extraversion
    agreeableness_score: conint(ge=0, le=5)
    agreeableness: Agreeableness
    conscientiousness_score: conint(ge=0, le=5)
    conscientiousness: Conscientiousness
    openness_to_experience_score: conint(ge=0, le=5)
    openness_to_experience: OpennessToExperience

# The base schema for dynamic creation
class PersonaDatasetQualityReviewBase(BaseModel):
    clarity: conint(ge=0, le=10)
    originality: conint(ge=0, le=10)
    coherence: conint(ge=0, le=10)
    diversity: conint(ge=0, le=10)
    realism: conint(ge=0, le=10)
    psychological_depth: conint(ge=0, le=10)
    hexaco_profile: HexacoProfile
    consistency: conint(ge=0, le=10)
    informativeness: conint(ge=0, le=10)
    ethical_considerations: conint(ge=0, le=10)
    demographic_fidelity: conint(ge=0, le=10)
    overall_score: conint(ge=0, le=10)
    # notes: Optional[str]  # Optional notes or comments about the review

client = OpenAI(api_key="YOUR_KEY_HERE")

import os
import pickle

# Try to load the reviews dictionary from disk, or create a new one if not present
reviews_dict = {}

for row in tqdm(dataset):  # Remove the select(range(2)) to review all entries
    uid = row["uuid"]
    if uid in reviews_dict:
        continue  # Skip if already reviewed

    # Dynamically create a schema with the UID set to the current row's UID as a default
    DynamicPersonaDatasetQualityReview = create_model(
        'DynamicPersonaDatasetQualityReview',
        UID=(str, uid),
        clarity=(conint(ge=0, le=10), ...),
        originality=(conint(ge=0, le=10), ...),
        coherence=(conint(ge=0, le=10), ...),
        diversity=(conint(ge=0, le=10), ...),
        realism=(conint(ge=0, le=10), ...),
        psychological_depth=(conint(ge=0, le=10), ...),
        hexaco_profile=(HexacoProfile, ...),
        consistency=(conint(ge=0, le=10), ...),
        informativeness=(conint(ge=0, le=10), ...),
        ethical_considerations=(conint(ge=0, le=10), ...),
        demographic_fidelity=(conint(ge=0, le=10), ...),
        overall_score=(conint(ge=0, le=10), ...),
        __base__=BaseModel
    )
    # If using Pydantic v2, call model_rebuild to ensure all references are resolved
    if hasattr(DynamicPersonaDatasetQualityReview, "model_rebuild"):
        DynamicPersonaDatasetQualityReview.model_rebuild()

    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {
                "role": "system",
                "content": (
                    "You are an expert reviewer evaluating the quality of entries in a dataset of police personas. "
                    "Your task is to assess the quality of the dataset entry itself, not the competency or character of the persona described. "
                    "Rate the following aspects from 0 (worst) to 5 (best):\n"
                    "- clarity: Is the persona description clear and understandable?\n"
                    "- originality: Does the entry avoid clichés and present a unique character?\n"
                    "- coherence: Is the information internally consistent and logically structured?\n"
                    "- diversity: Does the persona contribute to a diverse set of police profiles?\n"
                    "- realism: Does the persona feel plausible and authentic for a police context?\n"
                    "- psychological_depth: Does the entry provide meaningful insight into the persona's inner life, motivations, and psychological complexity? (Focus especially on this metric.)\n"
                    "- hexaco_profile: Please rate the persona on the six HEXACO dimensions (honesty_humility, emotionality, extraversion, agreeableness, conscientiousness, openness_to_experience), each from 0 (very low) to 5 (very high), and also rate all 24 HEXACO facet scores (sincerity, fairness, greed_avoidance, modesty, fearfulness, anxiety, dependence, sentimentality, social_self_esteem, social_boldness, sociability, liveliness, forgiveness, gentleness, flexibility, patience, organization, diligence, perfectionism, prudence, aesthetic_appreciation, inquisitiveness, creativity, unconventionality), each from 0 (very low) to 5 (very high), based on the information provided in the entry. Return these as a nested object.\n"
                    "- consistency: Are details about the persona consistent throughout?\n"
                    "- informativeness: Does the entry provide rich, relevant information about the persona?\n"
                    "- ethical_considerations: Is the entry free from harmful stereotypes or bias?\n"
                    "- demographic_fidelity: Is the persona plausible for the demographic data provided (e.g., a 22 year old should not be described as retiring with decades of experience)?\n"
                    "- overall_score: Your overall assessment of the dataset entry's quality.\n"
                    "Return only the scores in the specified schema, and include a UID string field (UID) for this entry. Remember: you are judging the quality of the dataset entry, not the police persona's job performance."
                ),
            },
            {
                "role": "user",
                "content": row["persona_string"],
            },
        ],
        text_format=DynamicPersonaDatasetQualityReview,
    )

    persona_review = response.output_parsed
    # Ensure the UID is set to the row's UID (in case the model doesn't return it correctly)
    persona_review.UID = uid
    # Upsert into the dictionary using UID as the key
    reviews_dict[uid] = persona_review

# Save the updated dictionary back to disk
import json

In [ ]:
final_reviews_dict = {item: value.model_dump() for item, value in reviews_dict.items()}

In [ ]:
import pickle

with open("really_final_list_of_reviews.json", "w") as f:
    json.dump(final_reviews_dict, f)


In [ ]:
#final_reviews_dict